<a href="https://colab.research.google.com/github/Mdadzd1/BotDetector/blob/main/ModelTrainer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is

In [2]:
import pandas as pd
from datasets import Dataset

# Assuming your CSV file is named 'bot_or_not_data.csv' and is in the Colab environment
try:
    df = pd.read_csv('training DS.csv')
except FileNotFoundError:
    print("Make sure your 'bot_or_not_data.csv' file is uploaded to Colab!")
    exit()

# Create a Hugging Face Dataset from the Pandas DataFrame
dataset = Dataset.from_pandas(df)

print(dataset)
print(dataset[0]) # Let's look at the first example

Dataset({
    features: ['question', 'text', 'label'],
    num_rows: 3132
})
{'question': 'What challenges do you think students face when learning about computer science', 'text': " I think students face challenges in learning computer science, such as limited prior exposure to programming concepts, struggling to understand complex algorithms, and difficulty in connecting theoretical concepts to real-world applications. Also, some students may feel intimidated by the subject's technical nature or feel that it's not relevant to their interests or future career goals.", 'label': 'ai'}


In [3]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Access the 'text' column as a list of strings, and convert each element to a string
    text_list = [str(text) for text in examples["text"]]
    # Tokenize the list of strings
    return tokenizer(text_list, truncation=True, padding="max_length", max_length=128)
    # Padding and max_length added to ensure all sequences have the same length

tokenized_dataset = dataset.map(tokenize_function, batched=True)

print(tokenized_dataset)
print(tokenized_dataset[0]) # Let's look at the first tokenized example

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/3132 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 3132
})
{'question': 'What challenges do you think students face when learning about computer science', 'text': " I think students face challenges in learning computer science, such as limited prior exposure to programming concepts, struggling to understand complex algorithms, and difficulty in connecting theoretical concepts to real-world applications. Also, some students may feel intimidated by the subject's technical nature or feel that it's not relevant to their interests or future career goals.", 'label': 'ai', 'input_ids': [101, 1045, 2228, 2493, 2227, 7860, 1999, 4083, 3274, 2671, 1010, 2107, 2004, 3132, 3188, 7524, 2000, 4730, 8474, 1010, 8084, 2000, 3305, 3375, 13792, 1010, 1998, 7669, 1999, 7176, 9373, 8474, 2000, 2613, 1011, 2088, 5097, 1012, 2036, 1010, 2070, 2493, 2089, 2514, 28028, 2011, 1996, 3395, 1005, 1055, 4087, 3267, 2030, 2514, 2008, 2009, 1005, 1055, 2025, 7882, 200

In [4]:
def label_to_int(example):
    if example["label"] == "human":
        return {"labels": 0}
    elif example["label"] == "ai":
        return {"labels": 1}
    else:
        return {"labels": -1} # Or handle other cases if necessary

labeled_dataset = tokenized_dataset.map(label_to_int)

# Remove the original 'label' column as we now have 'labels'
labeled_dataset = labeled_dataset.remove_columns(["label"])

print(labeled_dataset)
print(labeled_dataset[0]) # Let's look at the first labeled example

Map:   0%|          | 0/3132 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 3132
})
{'question': 'What challenges do you think students face when learning about computer science', 'text': " I think students face challenges in learning computer science, such as limited prior exposure to programming concepts, struggling to understand complex algorithms, and difficulty in connecting theoretical concepts to real-world applications. Also, some students may feel intimidated by the subject's technical nature or feel that it's not relevant to their interests or future career goals.", 'input_ids': [101, 1045, 2228, 2493, 2227, 7860, 1999, 4083, 3274, 2671, 1010, 2107, 2004, 3132, 3188, 7524, 2000, 4730, 8474, 1010, 8084, 2000, 3305, 3375, 13792, 1010, 1998, 7669, 1999, 7176, 9373, 8474, 2000, 2613, 1011, 2088, 5097, 1012, 2036, 1010, 2070, 2493, 2089, 2514, 28028, 2011, 1996, 3395, 1005, 1055, 4087, 3267, 2030, 2514, 2008, 2009, 1005, 1055, 2025, 7882, 2000, 2037, 5426,

In [5]:
train_test_split = labeled_dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['question', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2505
})
Dataset({
    features: ['question', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 627
})


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# Choose the same model name as the tokenizer
model_name = "distilbert-base-uncased"
num_labels = 2  # 'human' and 'ai'

# Load the pre-trained model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Define minimal training arguments
training_args = TrainingArguments(
    output_dir="./bot_detector_model",
    num_train_epochs=3,  # Or however many you want
    per_device_train_batch_size=16,
    report_to="none"  # Disable WandB
)

# Define a function to compute evaluation metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Create the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer
)

# Start the training process
trainer.train()

# Evaluate the best model on the evaluation set
eval_results = trainer.evaluate()
print("\nEvaluation Results:")
print(eval_results)

# Save the trained model and tokenizer
trainer.save_model("./trained_bot_detector")
tokenizer.save_pretrained("./trained_bot_detector")

print("\nTrained model and tokenizer saved to './trained_bot_detector'")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-6-248caeb902e9>:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


In [ ]:
import shutil
import os
from google.colab import files

folder_path = "./trained_bot_detector"
output_zip_file = "trained_bot_detector.zip"

# Create a zip file of the folder
shutil.make_archive(output_zip_file.replace(".zip", ""), 'zip', folder_path)

# Download the zip file
files.download(output_zip_file)